# Day 4: PHI Governance & De-Identification Engine

This notebook demonstrates the PySpark de-id module applying Format-Preserving Encryption (FPE) logic, deterministic date-shifting, and Safe Harbor 18 redaction.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sha2, date_add, date_sub, regexp_replace

spark = SparkSession.builder.appName("Day04_PHI_DeID").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark session created.")

In [ ]:
# Synthetic raw patient data
data = [
    ("P001", "123-45-6789", "2023-01-15", "Patient reported SSN is 123-45-6789 during intake."),
    ("P002", "987-65-4321", "2023-05-20", "No issues. Call 555-1234.")
]
columns = ["patient_id", "ssn", "encounter_date", "clinical_notes"]
raw_df = spark.createDataFrame(data, columns)

print("Raw Data (contains PHI):")
raw_df.show(truncate=False)

In [ ]:
def apply_deid(df):
    # 1. FPE/Hashing for SSN
    deid_df = df.withColumn("ssn_hashed", sha2(col("ssn"), 256)).drop("ssn")
    
    # 2. Date Shifting (-30 days)
    deid_df = deid_df.withColumn("encounter_date_shifted", date_sub(col("encounter_date"), 30)).drop("encounter_date")
    
    # 3. Text Redaction (Safe Harbor for SSN pattern)
    deid_df = deid_df.withColumn("clinical_notes_redacted", 
                                 regexp_replace(col("clinical_notes"), r"\d{3}-\d{2}-\d{4}", "[REDACTED_SSN]")) \
                     .drop("clinical_notes")
    
    return deid_df

deid_df = apply_deid(raw_df)

print("De-identified Data:")
deid_df.show(truncate=False)